# **RAG 기초**

## **1. 환경준비**

### (1) 구글 드라이브

* 구글 드라이브 폴더 생성
    * 새 폴더 `ai_agent`를 생성(이미 만들었다면 skip)
    * 제공 받은 파일을 업로드

* 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### (2) 라이브러리

* 필요한 라이브러리 설치

In [ ]:
!pip install -q langchain-openai langchain-community chromadb pymupdf langchain-chroma

* 라이브러리 로딩

In [ ]:
import pandas as pd
import numpy as np
import os

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader, CSVLoader, PyMuPDFLoader
from langchain_core.documents import Document

### (3) OpenAI API Key 확인

In [ ]:
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

path = '/content/drive/MyDrive/ai_agent/'

# API 키 로드 및 환경변수 설정
load_api_keys(path + 'api_key.txt')

* ⚠️ key가 제대로 보이는지 확인하세요.

In [ ]:
print(os.environ['OPENAI_API_KEY'][:40])

## **2. Vector DB 구성① : 기본 방법**

### (1) Loader

* TextLoader
    - 일반적인 텍스트 파일(.txt)에서 데이터를 불러오는 역할
    - txt 파일을 읽고, 이를 문서 객체(Document)로 변환


In [ ]:
from langchain_community.document_loaders import TextLoader

# 텍스트 파일 경로 지정
file_path = "상록수.txt"

# TextLoader를 이용하여 문서 로드
loader = TextLoader(path + file_path)
txt_docs = loader.load()

# 로드된 문서 출력
print(txt_docs)

In [ ]:
txt_docs

* 로딩된 데이터의 타입과 길이를 살펴 봅시다.

In [ ]:
print(type(txt_docs))
print(len(txt_docs))
print(type(txt_docs[0]))

In [ ]:
txt_docs[0].page_content[:1000]

* Document
    - LangChain에서 텍스트 데이터(문서)를 구조적으로 표현하는 기본 단위
    - 주요 속성
        - metadata : 문서의 출처, 태그, 카테고리 등의 부가 정보
        - page_content : 문서의 실제 텍스트 내용
    - Loader로 로딩한 후 저장하거나, 직접 Document 생성 가능


In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="이것은 LangChain의 Document 객체 예제입니다.",
    metadata={"source": "sample.txt", "category": "example"}
)
doc

In [ ]:
doc.metadata
#doc.metadata['source']

In [ ]:
doc.page_content

### (2) Splitter

- 긴 문서를 작은 단위인 청크(chunk)로 나누는 텍스트 분리 도구
    - 텍스트를 분리하는 작업 : 청킹(chunking)
        - LLM 모델의 입력 토큰의 개수가 정해져 있기 때문
        - 텍스트가 너무 긴 경우에는 핵심 정보 이외에 불필요한 정보들이 많이 포함 → RAG 품질 저하 요인
        - 핵심 정보가 유지될 수 있는 적절한 크기로 나누는 것이 매우 중요

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

* 글자 단위

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 100,
    separator = '',   # 어떤 기준 없이 무조건 500자 단위로 나누라
)

# text 로딩한 결과가 Document이므로, split_documents로 분할
split_texts = text_splitter.split_documents(txt_docs)

# 결과 확인
for i, chunk in enumerate(split_texts[:3]):  # 처음 3개 청크만 출력
    print('-'*200)
    print(f"[청크 {i+1}, 길이 {len(chunk.page_content)}]\n{chunk.page_content}\n")

* 문장 단위

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separator="."  # 마침표 기준으로 분할(문장 단위로 분할)
)

split_texts = text_splitter.split_documents(txt_docs)

# 결과 확인
for i, chunk in enumerate(split_texts[:3]):  # 처음 3개 청크만 출력
    print('-'*200)
    print(f"[청크 {i+1}, 길이 {len(chunk.page_content)}]\n{chunk.page_content}\n")

In [ ]:
len(split_texts)

In [ ]:
split_texts[:3]

* 줄 바꿈 단위

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 100,
    separator = '\n',   # \n 줄바꿈 문자 기준으로 자르기
)

split_texts = text_splitter.split_documents(txt_docs)

# 결과 확인
for i, chunk in enumerate(split_texts[:3]):  # 처음 3개 청크만 출력
    print('-'*200)
    print(f"[청크 {i+1}, 길이 {len(chunk.page_content)}]\n{chunk.page_content}\n")

#### 실습🔥
* 다음의 옵션을 자유롭게 조절하며 청크의 크기가 어떻게 나뉘는지 살펴 봅시다.
    * chunk_size
    * chunk_overlap
    * separator

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size = 100,       # 조절
    chunk_overlap  = 50,    # 조절
    separator = '',         # 조절
)

split_texts = text_splitter.split_documents(txt_docs)

# 결과 확인
for i, chunk in enumerate(split_texts[:3]):  # 처음 3개 청크만 출력
    print('-'*200)
    print(f"[청크 {i+1}, 길이 {len(chunk.page_content)}]\n{chunk.page_content}\n")

### **(3) Embedding & Store**

#### **1) Embedding model 선언**

In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# 임베딩 벡터
sample_text1 = "농촌 계몽운동은 한국 근대화의 중요한 과정이었다."
vector1 = embedding_model.embed_query(sample_text1)

# 벡터 길이 및 일부 값 확인
print(f"임베딩 벡터 길이: {len(vector1)}")
print(f"첫 10개 벡터 값: {vector1[:10]}")

#### **2) ChromaDB 저장**

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separator="."  # 문장 단위로 분할
)

split_texts = text_splitter.split_documents(txt_docs)

In [ ]:
# ChromaDB를 만들면서 저장
db_txt = Chroma.from_documents(split_texts, embedding_model,
                               persist_directory="./db01")

### **(4) 유사도 검색**

#### **1) 벡터DB에서 유사도 기반 검색**

In [ ]:
query = "농촌 계몽운동에 대한 내용"
retrieved_docs = db_txt.similarity_search(query, k=3)

# 벡터DB에서 검색한 결과는 list[Document,...]
retrieved_docs

In [ ]:
# 결과 출력
print("[검색 결과]")
print('='*200)
for doc in retrieved_docs:
    print(doc.page_content)
    print('-'*200)

#### **2) 유사도 점수도 출력하면서 검색**
* **Chroma DB**나 **FAISS**의 기본 거리계산법 : L2 거리(유클리드 거리)  
$$\|a - b\| = \sqrt{\sum(a_i - b_i)^2}$$

* 단, Chroma DB에서는 루트 계산(sqrt)을 하지 않은 값으로 처리

In [ ]:
query = "농촌 계몽운동에 대한 내용"
retrieved_docs = db_txt.similarity_search_with_score(query, k=3)

# 결과 형태 확인
retrieved_docs

In [ ]:
# 결과를 하나씩 출력
for doc, score in retrieved_docs:
    print(f"Score(L2 ** 2): {score:.4f}")
    print(f"Content: {doc.page_content[:100]}")
    print("-" * 50)

#### 실습🔥

① 몇 가지 질문을 던저서 적절한 문서를 가져오는지 확인해 봅시다.(유사도 거리도 함께 표시)

② 유사도 거리 기준을 임의로 주고, 유사도 거리 이내인 문서만 검색하도록 합시다.

## **3. Vector DB 구성② : 다양한 방법**

### **(1) Loader**

#### **1) PDF loader**
- PDFLoader는 PDF 문서를 로딩하여 페이지 별 Document로 변환해 주는 도구
- 청킹 벡터화하여 검색 가능하도록 만듦.


In [ ]:
# PDF 파일 로드
pdf_path = "롯데쇼핑 서비스 이용약관.pdf"
pdf_loader = PyMuPDFLoader(path + pdf_path)

# 문서 로드 실행
doc_pdf = pdf_loader.load()

# 출력 확인
print(f"총 {len(doc_pdf)} 개의 페이지가 로드됨")

#### 실습🔥
* doc_pdf를 열어 구조를 살펴봅시다.
    * type
    * 길이
    * 열 번 째 값
        * metadata
        * page_content
* pdf 파일 원본도 열어서 비교해 봅시다.

#### **2) csv 로더**
- 각 행을 하나의 Document 객체로 변환


In [ ]:
data = pd.read_csv(path+'sample.csv')
data.head()

In [ ]:
data.shape

In [ ]:
# CSV 파일 로드
csv_path = "sample.csv"
csv_loader = CSVLoader(file_path= path + csv_path)

# 문서 로드 실행
doc_csv = csv_loader.load()

# 첫 번째 행 출력
print(f"총 {len(doc_csv)} 개의 행이 로드됨")
print(doc_csv[0].page_content)
print(doc_csv[0].metadata)

### **(2) Splitter**

#### **1) PDF**

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 100,
    separator = '\n',   # \n 줄바꿈 문자 기준으로 자르기,
)

split_pdf = text_splitter.split_documents(doc_pdf)
print(f"원본 문서 개수: {len(doc_pdf)}")
print(f"분할된 청크 개수: {len(split_pdf)}")
print(f"첫 번째 청크:\n{split_pdf[0].page_content}")

In [ ]:
split_pdf[3].metadata

In [ ]:
split_pdf[0].page_content

#### **2) CSV**
csv는 이미 행 별로 document 분리 되어 있으므로, 별도로 split 필요하지 않음

### **(3) Embedding & Store**

#### **PDF**

* Embedding model 선언

In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

* DB 저장

In [ ]:
# ChromaDB를 만들면서 저장
db_pdf = Chroma.from_documents(split_pdf, embedding_model,
                               persist_directory="./db02")

In [ ]:
db_pdf

* DB에 추가 입력

In [ ]:
# 상록수 DB에 문서를 추가해 봅시다.
new_docs = [
    Document(page_content="조선시대의 교육 제도는 성균관 중심이었다.",
             metadata={"source": "추가"}),
    Document(page_content="한국 전통 사회에서 글을 읽는 능력은 권력의 상징이었다.",
             metadata={"source": "추가"})
]

db_txt.add_documents(new_docs)

### **(4) 유사도 검색**

* 유사도 높은 문서 조회

In [ ]:
query = "배송완료 후 며칠 이내에 반품 신청할 수 있나요?"
retrieved_docs = db_pdf.similarity_search(query, k=3)

# 결과 출력
print("검색 결과:")
for doc in retrieved_docs:
    print(doc.page_content)
    print('-'*200)

* 유사도 점수도 함께 조회

In [ ]:
# 검색
query = "배송완료 후 며칠 이내에 반품 신청할 수 있나요?"
retrieved_docs = db_pdf.similarity_search_with_score(query, k=3)

# 점수 추출 + 변환
for doc, score in retrieved_docs:
    print('유사도 점수 :', score)
    print('문서 내용 :', doc.page_content)
    print('-'*200)

* 임계값으로 문서 필터링

In [ ]:
# 유사도 임계값 1 이하인 문서만 필터링
threshold = 1
query = "배송완료 후 며칠 이내에 반품 신청할 수 있나요?"

def filtered_db_search(query, threshold, k=3):
    text_list = []
    results = db_pdf.similarity_search_with_score(query, k=k)

    for doc, score in results :
        if score <= threshold :
            text_list.append(doc.page_content)  # 검색된 Document를 리스트에 추가
    return '\n\n'.join(text_list)   # 리스트의 각 항목을 두번 줄바꿈(\n\n)하며 붙이기

# 사용 예시
docs = filtered_db_search(query, threshold)
print(docs)

### (5) 실습🔥
doc_csv를 벡터DB로 저장시키기

* loader : 이미 수행됨 : doc_csv
* splitter : 한 행을 하나의 chunk로 지정하는 것이라면, 별도 split 필요 없음.
* doc_csv 를 chroma db에 저장하기
    * db name : db_csv

In [ ]:
# DB 저장


* 두개의 문서 추가
    * 문서1
        * 구분 : 지적재산권
        * 내용 : 여기에 지적재산권 문제도 있다. 누구나 사용할 수 있도록 만들어진 것이 오픈소스 소프트웨어지만 세밀하게 들여다보면 각 소프트웨어 별로 라이선스가 다르다. 재배포를 허용하는 것도 있고 아닌것도 있으며 하나의 소프트웨어만 작동할 때는 무료지만 다른 기능을 사용하려면 유료로 전환되는 것도 있다.
    * 문서2
        * 구분 : 보안
        * 내용 : 개개인이 분산 방식으로 협업하는 오픈소스는 보안이 취약한 분야가 생길 가능성이 크다. 그리고 이를 악용하는 개발자도 당연히 있을 수 있다. 그리고 오픈소스를 사용해 프로그램을 만들면 참고한 오픈소스에 종속되는 경우가 많기에 오픈소스에 문제가 생기면 프로그램 자체에 중대한 오류가 발생할 가능성이 높다는 위험이 있다.

* 유사도 검색 시도하기

## **4. 벡터DB 살펴보기**

### (1) 전체 데이터

* 전체 문서 수

In [ ]:
print(db_pdf._collection.count())

* 전체 데이터 가져오기

In [ ]:
# 전체 데이터 가져오기
result = db_pdf.get()

# 포함된 키: ids, documents, metadatas, embeddings(기본 제외)
print(result.keys())       # dict_keys(['ids', 'embeddings', 'documents', 'metadatas'])
print(result['ids'])       # 문서 ID 목록
print(result['documents']) # 청크 텍스트 목록
print(result['metadatas']) # 메타데이터 목록 (페이지 번호, 소스 파일 등)

### (2) 조건 조회
* id 조회
* 메타데이터 필터링
* 필요 정보 조회

* id로 특정 문서 조회

In [ ]:
ids = result['ids'][:3]
result = db_pdf.get(ids=ids)
print(result['documents'])

* meta data로 필터링

In [ ]:
result = db_pdf.get(where={"page": 5})
print(result)

* 필요한 데이터 조회

In [ ]:
# 특정 키 값 포함 + 필터링
result = db_pdf.get(include=["embeddings", "documents"], where={"page": 5})
result

### (3) 실습🔥
* 실습 3-(5)의 DB를 살펴 봅시다.
    * 전체 조회
    * 메타데이터에서 row 번호 지정하여, document와 embedding 벡터 조회

In [ ]:
# 전체 데이터 가져오기


# 포함된 키: ids, documents, metadatas, embeddings(기본 제외)

